# MediaPipe → SMPL mapping: interactive check on one photograph

Runs the **current, tested** MediaPipe-to-SMPL initialization on a photo of your choice.
The entry point is **`initialize_smpl_from_mediapipe()`**, which returns the full-body SMPL pose
(`global_orient` from the torso, plus shoulder/elbow/hip/knee rotations in `body_pose`). It is run in two variants:

- **A (swing-only):** minimal rotations that match each bone direction. Twist about each bone is not estimated.
- **B (twist + hinge):** elbow/knee as a pure hinge, with shoulder/hip twist solved from the bend plane. B falls back to A (with a reason) when a limb is nearly straight (arms < 15°, legs < 20°) or the hip twist exceeds 60°.

The **primary result is the twist variant (B)**; A is computed only for comparison.
All mathematics lives in `src/smpl_utils.py`. This notebook only calls it, plus the reporting and plotting helpers in
`scripts/validate_global_orient.py` and `scripts/validate_arm_pose.py`. `betas`, `transl`, spine, collars, neck, head, wrists, hands, ankles and feet stay at zero.

### Setup (once)
1. **Python environment:** from `3d-pose-diff-render/`:
   ```bash
   python3 -m venv .venv
   .venv/bin/pip install -r requirements.txt ipykernel nbformat
   ```
   `ipykernel` lets VS Code use `.venv` as a kernel; `nbformat` lets Plotly render inline.
2. **VS Code kernel:** open this notebook, click **Select Kernel → Python Environments →** `3d-pose-diff-render/.venv/bin/python`.
3. **Model files** (not in the repository, see README):
   - Chumpy-free SMPL neutral model: set the environment variable `SMPL_MODEL_PATH` *before starting VS Code*, or edit `SMPL_MODEL_PATH` in the configuration cell.
   - MediaPipe pose model: `~/models/mediapipe/pose_landmarker_heavy.task` by default (or set `POSE_MODEL_PATH`).

### Execution order
Run the cells **top to bottom** the first time (**Run All** works).
To try another photograph, change `IMAGE_PATH` in the configuration cell and run it plus everything below it.
The SMPL model is loaded once and reused.


## 1. Configuration: edit this cell

In [ ]:
import os
from pathlib import Path

# Photograph to analyse (any JPEG/PNG with one clearly visible person).
# Leave as-is to read the MP_IMAGE_PATH environment variable instead.
IMAGE_PATH = os.environ.get("MP_IMAGE_PATH", "path/to/your/photo.jpg")

# Model files (see the setup notes above). Defaults avoid personal directories.
SMPL_MODEL_PATH = os.environ.get("SMPL_MODEL_PATH", "path/to/SMPL_NEUTRAL.pkl")
POSE_MODEL_PATH = os.environ.get("POSE_MODEL_PATH",
                                 str(Path.home() / "models" / "mediapipe" / "pose_landmarker_heavy.task"))

LIMBS = "all"        # "arms", "legs" or "all"
SIDES = "both"       # "left", "right" or "both" (the person's own side)
SHOW_MESH = None     # None (toggle in the legend), "B" (full-body twist pose) or "A" (swing-only): show that mesh at once
QUIET_MEDIAPIPE = True  # hide MediaPipe's native INFO/WARNING log lines during detection

# Optional: save metrics JSON, the 2D overlay and the 3D HTML. Must be outside the repository.
SAVE_OUTPUTS = False
OUT_DIR = Path(os.environ.get("SMPL_MAPPING_OUT_DIR", Path.home() / "cv_outputs" / "smpl_mapping"))

## 2. Imports and checks

In [ ]:
import sys
import json

import numpy as np

# Locate the project root (the folder that contains src/ and scripts/), wherever the kernel started.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "smpl_utils.py").exists())
for p in (ROOT, ROOT / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src.smpl_utils import (SMPL_JOINT_NAMES, initialize_smpl_from_mediapipe, load_smpl, mediapipe_to_arrays,
                            mediapipe_world_to_camera_aligned, smpl_forward, smpl_rest_torso_orientation)
import validate_global_orient as vgo   # MediaPipe detection, 2D overlay, global-orientation checks
import validate_arm_pose as vap       # limb diagnostics and the 3D figure

def _require_file(path, what, hint):
    if not Path(path).expanduser().is_file():
        raise FileNotFoundError(f"{what} not found: {path}\n{hint}")

_require_file(IMAGE_PATH, "Photograph", "Set IMAGE_PATH in the configuration cell (or the MP_IMAGE_PATH environment variable).")
_require_file(SMPL_MODEL_PATH, "SMPL model", "Set SMPL_MODEL_PATH to the Chumpy-free SMPL_NEUTRAL.pkl (see README).")
_require_file(POSE_MODEL_PATH, "MediaPipe pose model", "Download pose_landmarker_heavy.task (see README) or set POSE_MODEL_PATH.")
if LIMBS not in ("arms", "legs", "all") or SIDES not in ("left", "right", "both"):
    raise ValueError("LIMBS must be arms/legs/all and SIDES left/right/both")

KINDS = {"arms": ("arm",), "legs": ("leg",), "all": ("arm", "leg")}[LIMBS]
LIMB_LIST = [(limb, side) for limb in KINDS for side in (("left", "right") if SIDES == "both" else (SIDES,))]
print("project root:", ROOT)
print("photo:       ", IMAGE_PATH)
print("limbs:       ", LIMB_LIST)

## 3. Load SMPL (cached between photos)

In [ ]:
import torch

if globals().get("_SMPL_CACHE", (None,))[0] != SMPL_MODEL_PATH:
    _model = load_smpl(SMPL_MODEL_PATH)
    with torch.no_grad():
        _rest = smpl_forward(_model)["joints"][0].double().numpy()
    _SMPL_CACHE = (SMPL_MODEL_PATH, _model, smpl_rest_torso_orientation(_model), _rest)
_, MODEL, R_REST, REST_JOINTS = _SMPL_CACHE
print("SMPL loaded:", SMPL_MODEL_PATH, "| num_betas =", MODEL.num_betas)

## 4. Photograph and MediaPipe landmarks

In [ ]:
import contextlib

@contextlib.contextmanager
def _quiet_native_stderr(enabled=True):
    """Silence MediaPipe's C++ log lines (they bypass Python logging). Python exceptions are unaffected."""
    if not enabled:
        yield
        return
    sys.stderr.flush()
    saved, devnull = os.dup(2), os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(saved)
        os.close(devnull)

rgb = vgo.read_rgb(str(Path(IMAGE_PATH).expanduser()))
with _quiet_native_stderr(QUIET_MEDIAPIPE):
    result = vgo.detect_pose(rgb, str(Path(POSE_MODEL_PATH).expanduser()))
try:
    world_raw, pixels, vis = mediapipe_to_arrays(result, rgb.shape[:2])
except ValueError as e:
    raise RuntimeError(f"MediaPipe found no usable pose in this photo: {e}") from None
world = mediapipe_world_to_camera_aligned(world_raw).astype(np.float64)
print(f"image {rgb.shape[1]}x{rgb.shape[0]} px | landmarks with visibility < 0.5: "
      f"{[vgo.MEDIAPIPE_LANDMARK_NAMES[i] for i in np.flatnonzero(vis < 0.5)] or 'none'}")

In [ ]:
import cv2
import matplotlib.pyplot as plt

overlay = cv2.cvtColor(vgo.draw_overlay(rgb, {"pixels": pixels, "vis": vis}), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(6, 6 * overlay.shape[0] / overlay.shape[1]))
plt.imshow(overlay)
plt.title("MediaPipe landmarks (green: visibility >= 0.5, red: lower)")
plt.axis("off")
plt.show()

## 5. Global orientation and convention checks
`A` is near zero by construction; the other checks are diagnostics (see `scripts/validate_global_orient.py`).

In [ ]:
from IPython.display import Markdown, display

go_metrics, _ = vgo.analyze(result, rgb.shape[:2], MODEL, R_REST)
E, F, C, D = (go_metrics[k] for k in ("E_orientation", "F_bone_residuals_deg", "C_left_right", "D_anatomical_forward"))
display(Markdown(f"""
| quantity | value |
|---|---|
| SMPL `global_orient` (axis-angle) | `{np.round(E["global_orient_axis_angle"], 4).tolist()}` ({E["global_orient_deg"]:.1f} deg) |
| torso yaw / spine tilt | {E["torso_yaw_deg"]:.1f} deg / {E["spine_tilt_from_image_up_deg"]:.1f} deg |
| A: torso frame error (SMPL vs MediaPipe) | {go_metrics["A_orientation_error_deg"]:.1e} deg |
| F: shoulder-line / hip-line residual | {F["shoulder_line"]:.1f} / {F["hip_line"]:.1f} deg |
| B: corr(world x, pixel x) / corr(world y, pixel y) | {go_metrics["B_correlation"]["world_x_vs_pixel_x"]:.3f} / {go_metrics["B_correlation"]["world_y_vs_pixel_y"]:.3f} |
| C: left/right labels consistent with facing | {C["consistent_with_facing"]} (facing camera: {C["facing_camera_from_world"]}) |
| D: nose and toes in front of the torso | {D["all_positive"]} |
| torso landmarks with low visibility | {go_metrics["low_visibility_torso_landmarks"] or "none"} |
"""))

## 6. Full-body pose: `initialize_smpl_from_mediapipe()`
`POSE` (variant `"twist"`) is the result to inspect; `POSE_SWING` is only for comparison.
The table shows, per limb, whether the twist + hinge solution was used or the limb fell back to swing-only (and why).

In [ ]:
POSE = initialize_smpl_from_mediapipe(
    world, REST_JOINTS, limbs=LIMB_LIST, variant="twist",
    straight_arm_deg=vap.STRAIGHT_ARM_DEG, straight_leg_deg=vap.STRAIGHT_LEG_DEG,
    max_hip_twist_deg=vap.MAX_HIP_TWIST_DEG)
POSE_SWING = initialize_smpl_from_mediapipe(world, REST_JOINTS, limbs=LIMB_LIST, variant="swing")

assert np.allclose(POSE["global_orient"], go_metrics["E_orientation"]["global_orient_axis_angle"])
rows = ["| limb | method used | reason (if fallback) |", "|---|---|---|"]
for name, d in POSE["limbs"].items():
    method = "**twist + hinge**" if d["twist_resolved"] else "swing-only (fallback)"
    rows.append(f"| {name} | {method} | {d['reason'] or ''} |")
bp = POSE["body_pose"]
joints = ["| SMPL joint | body_pose slots | axis-angle | rotation (deg) |", "|---|---|---|---|"]
for j in range(1, 24):
    r = bp[3 * (j - 1): 3 * j]
    if np.any(r != 0):
        joints.append(f"| {SMPL_JOINT_NAMES[j]} ({j}) | {3 * (j - 1)}-{3 * j - 1} | `{np.round(r, 4).tolist()}` "
                      f"| {np.degrees(np.linalg.norm(r)):.1f} |")
n_zero = sum(1 for j in range(1, 24) if not np.any(bp[3 * (j - 1): 3 * j]))
display(Markdown(
    f"**global_orient** `{np.round(POSE['global_orient'], 4).tolist()}` "
    f"({np.degrees(np.linalg.norm(POSE['global_orient'])):.1f} deg)\n\n" + "\n".join(rows)
    + "\n\n**Non-zero body_pose joints**\n\n" + "\n".join(joints)
    + f"\n\nThe other {n_zero} joints are exactly zero (not estimated yet)."))

## 7. Diagnostics: A (swing-only) vs B (full-body twist pose)
- **direction error:** angle between SMPL and MediaPipe bone directions (proximal / distal bone)
- **off-hinge:** how far the elbow/knee rotation axis is from its anatomical hinge (0 = pure hinge)
- **hinge vs n:** angle between the SMPL hinge axis and the observed bend-plane normal (non-zero is expected, because of the rest-pose offset)
- **twist** (ψ): shoulder/hip rotation about the upper arm/thigh; **φ**: hinge angle
- **foot yaw diff** (legs): independent check, since the foot is not used by the solver

In [ ]:
limb_metrics, limb_arrays = vap.analyze_initialization(
    world, vis, MODEL, LIMB_LIST, {"A": POSE_SWING, "B": POSE}, REST_JOINTS)

def _f(v, fmt=".1f"):
    return "–" if v is None else format(v, fmt)

rows = ["| limb | vis min | bend | A dir err | A off-hinge | B dir err | B resolved | B off-hinge | twist ψ | hinge φ | hinge vs n | foot yaw diff A/B |",
        "|---|---|---|---|---|---|---|---|---|---|---|---|"]
notes = []
for name, e in limb_metrics["limbs"].items():
    A, B = e["A_swing_only"], e["B_twist_corrected"]
    dA, dB = A["direction_residual_deg"], B["direction_residual_deg"]
    foot = (f'{A["foot"]["yaw_diff_deg"]:.1f} / {B["foot"]["yaw_diff_deg"]:.1f}' if "foot" in A else "–")
    rows.append(
        f'| {name} | {min(e["visibility"].values()):.2f} | {e["bend_mediapipe_deg"]:.1f} '
        f'| {dA["proximal_bone"]:.0e} / {dA["distal_bone"]:.0e} | {A["off_hinge_deg"]:.1f} '
        f'| {dB["proximal_bone"]:.0e} / {dB["distal_bone"]:.0e} | {"yes" if B["twist_resolved"] else "**no (A)**"} '
        f'| {_f(B["off_hinge_deg"], ".1e") if B["twist_resolved"] else _f(B["off_hinge_deg"])} | {_f(B["psi_deg"])} | {_f(B["phi_deg"])} '
        f'| {B["hinge_axis_vs_bend_normal_deg"]:.1f} | {foot} |')
    if B["reason"]:
        notes.append(f"- **{name}**: fallback to A: {B['reason']}")
    if e["low_visibility"]:
        notes.append(f"- **{name}**: low-visibility landmarks {e['low_visibility']}")
    if B.get("phi_plausible") is False:
        notes.append(f"- **{name}**: hinge angle outside the plausible range")
display(Markdown("\n".join(rows) + ("\n\n" + "\n".join(notes) if notes else "")))
print("all angles in degrees; thresholds: arms", vap.STRAIGHT_ARM_DEG, "| legs", vap.STRAIGHT_LEG_DEG,
      "| max hip twist", vap.MAX_HIP_TWIST_DEG)

## 8. MediaPipe vs SMPL bone directions (full-body pose)
Unit bone directions in the camera-aligned frame (+X right, +Y down, +Z away from the camera), for MediaPipe and for the
SMPL skeleton posed with `POSE`, and the angle between them.

In [ ]:
J_pose = limb_arrays["J"]["B"]
rows = ["| limb | bone | MediaPipe direction | SMPL direction | angle (deg) |", "|---|---|---|---|---|"]
for limb, side in LIMB_LIST:
    (m_s, m_e, m_w), (s_s, s_e, s_w) = vap.INDICES[(limb, side)]
    bones = (("upper arm", "forearm") if limb == "arm" else ("thigh", "shin"))
    for bone, (ma, mb), (sa, sb) in zip(bones, ((m_s, m_e), (m_e, m_w)), ((s_s, s_e), (s_e, s_w))):
        u = world[mb] - world[ma]; u = u / np.linalg.norm(u)
        v = J_pose[sb] - J_pose[sa]; v = v / np.linalg.norm(v)
        angle = np.degrees(np.arctan2(np.linalg.norm(np.cross(u, v)), u @ v))
        rows.append(f"| {limb}_{side} | {bone} | `{np.round(u, 3).tolist()}` | `{np.round(v, 3).tolist()}` | {angle:.1e} |")
display(Markdown("\n".join(rows)))

## 9. Interactive 3D view
Rotate and zoom with the mouse. **B is the full-body pose from `initialize_smpl_from_mediapipe()`.** The dropdown (top left) switches **both / A / B**; click legend entries to toggle traces, including the SMPL meshes.
Arrows: cyan dotted = observed bone directions, orange = observed bend normal, olive/purple = A/B hinge axes, brown = MediaPipe foot.
Each body is centred on its own hip midpoint for display only (no camera translation is estimated).

In [ ]:
fig = vap.build_figure(limb_arrays, limb_metrics, LIMB_LIST, show_mesh=SHOW_MESH)
fig.update_layout(height=750)
fig.show()

## 10. Optional: save outputs

In [ ]:
if SAVE_OUTPUTS:
    out = Path(OUT_DIR).expanduser().resolve() / Path(IMAGE_PATH).stem
    if ROOT.resolve() in (out, *out.parents):
        raise ValueError(f"OUT_DIR must be outside the repository: {out}")
    out.mkdir(parents=True, exist_ok=True)
    (out / "global_orient_metrics.json").write_text(json.dumps(go_metrics, indent=2))
    (out / f"smpl_pose_{LIMBS}_{SIDES}.json").write_text(json.dumps(
        {"global_orient": POSE["global_orient"].tolist(), "body_pose": POSE["body_pose"].tolist(),
         "limbs": POSE["limbs"]}, indent=2, default=list))
    (out / f"limb_metrics_{LIMBS}_{SIDES}.json").write_text(json.dumps(limb_metrics, indent=2))
    cv2.imwrite(str(out / "landmarks_2d.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
    fig.write_html(str(out / f"limb_pose_3d_{LIMBS}_{SIDES}.html"), include_plotlyjs=True)
    print("saved to", out)
else:
    print("SAVE_OUTPUTS is False: nothing written")

## Trying another photograph
1. Change `IMAGE_PATH` (and optionally `LIMBS`, `SIDES`, `SHOW_MESH`) in the configuration cell.
2. Run the configuration cell and every cell below it (or **Run All**). SMPL is reused from the cache.

Interpretation tips: B is only expected to differ from A when a limb is clearly bent. A large shoulder twist can be correct (e.g. hands in pockets).
Knee bends seen from the front lie mostly along the camera depth axis, which is the noisiest MediaPipe direction.